In [23]:
%pip install datasets transformers
%pip install lm-eval

from pathlib import Path
import torch
from torch import nn
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 84.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ea69386b08166a00c94dd1dd16871a78ca4798e4668e7ec33a51f75537be67dc
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16862 sha256=9bb25f9135446cc4102f1bea4595a29735ed7f02f6f8b6c4efe8247469d64fbc
  Stored in directory: /root/.cache/pip/wheels/

In [ ]:

def load_model_and_tokenizer(
    model_id,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=False,
):
    """Load a Hugging Face causal LM and its tokenizer for pruning experiments."""
    print(f"Loading tokenizer: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, trust_remote_code=trust_remote_code
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch_dtype,
        device_map=device_map,
        trust_remote_code=trust_remote_code,
    )
    model.eval()
    model.config.use_cache = False
    return model, tokenizer


def get_input_device(model):
    """Return the device where input_ids should be placed."""
    return next(model.parameters()).device


def get_decoder_layers(model):
    """Find the repeated decoder/language-model blocks in common causal LMs."""
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    raise ValueError("Could not find decoder layers for this model.")


def get_text_model(model):
    """Return the inner text model from a CausalLM wrapper."""
    if hasattr(model, "model"):
        return model.model
    raise ValueError("Expected the loaded model to have an inner .model text module.")


def load_c4_calibration(
    tokenizer,
    n_samples,
    sequence_length,
    seed=0,
):
    """Sample fixed-length C4 token sequences for calibration.

    Wanda/SparseGPT-style calibration uses unlabeled text only. We use the first C4
    English training shard and cut random contiguous spans from random documents.
    """
    print(
        "Loading C4 calibration data from allenai/c4 first train shard "
        f"(samples={n_samples}, sequence_length={sequence_length})..."
    )
    dataset = load_dataset(
        "allenai/c4",
        data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
        split="train",
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    samples = []
    attempts = 0
    max_attempts = n_samples * 100
    while len(samples) < n_samples and attempts < max_attempts:
        attempts += 1
        row_idx = torch.randint(0, len(dataset), (1,), generator=generator).item()
        text = dataset[row_idx]["text"]
        input_ids = tokenizer(
            text, return_tensors="pt", add_special_tokens=False
        ).input_ids

        if input_ids.shape[1] < sequence_length:
            continue

        max_start = input_ids.shape[1] - sequence_length
        start = torch.randint(0, max_start + 1, (1,), generator=generator).item()
        samples.append(input_ids[:, start : start + sequence_length])
        if len(samples) % 10 == 0:
            print(f"  collected calibration sample {len(samples)}/{n_samples}")

    if len(samples) != n_samples:
        raise RuntimeError(f"Collected {len(samples)} samples, expected {n_samples}.")

    print(f"Collected {len(samples)} calibration sequences of length {sequence_length}.")
    return samples

def load_hybrid_calibration(
    tokenizer,
    n_samples,
    sequence_length,
    seed=0,
):
    """Sample a 50-50 mix of C4 general English text and GSM8K math word problems.

    GSM8K samples are constructed by concatenating multiple random QA pairs until
    the sequence length is met. C4 samples are cut from random documents.
    """
    print(
        f"Loading 50-50 Hybrid (C4 + GSM8K) calibration data "
        f"(samples={n_samples}, sequence_length={sequence_length})..."
    )
    
    n_c4 = n_samples // 2
    n_gsm = n_samples - n_c4

    generator = torch.Generator()
    generator.manual_seed(seed)

    # 1. Load C4 dataset
    c4_dataset = load_dataset(
        "allenai/c4",
        data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
        split="train",
    )

    c4_samples = []
    attempts = 0
    max_attempts = n_c4 * 100
    while len(c4_samples) < n_c4 and attempts < max_attempts:
        attempts += 1
        row_idx = torch.randint(0, len(c4_dataset), (1,), generator=generator).item()
        text = c4_dataset[row_idx]["text"]
        input_ids = tokenizer(
            text, return_tensors="pt", add_special_tokens=False
        ).input_ids

        if input_ids.shape[1] < sequence_length:
            continue

        max_start = input_ids.shape[1] - sequence_length
        start = torch.randint(0, max_start + 1, (1,), generator=generator).item()
        c4_samples.append(input_ids[:, start : start + sequence_length])
        print(f"  collected C4 calibration sample {len(c4_samples)}/{n_c4}")

    if len(c4_samples) != n_c4:
        raise RuntimeError(f"Collected {len(c4_samples)} C4 samples, expected {n_c4}.")

    # 2. Load GSM8K dataset
    gsm_dataset = load_dataset("gsm8k", "main", split="train")

    gsm_samples = []
    attempts = 0
    max_attempts = n_gsm * 100
    while len(gsm_samples) < n_gsm and attempts < max_attempts:
        attempts += 1
        current_text = ""
        while True:
            row_idx = torch.randint(0, len(gsm_dataset), (1,), generator=generator).item()
            row = gsm_dataset[row_idx]
            current_text += f"Question: {row['question']}\nAnswer: {row['answer']}\n\n"
            
            input_ids = tokenizer(
                current_text, return_tensors="pt", add_special_tokens=False
            ).input_ids
            
            if input_ids.shape[1] >= sequence_length:
                max_start = input_ids.shape[1] - sequence_length
                start = torch.randint(0, max_start + 1, (1,), generator=generator).item()
                gsm_samples.append(input_ids[:, start : start + sequence_length])
                print(f"  collected GSM8K calibration sample {len(gsm_samples)}/{n_gsm}")
                break

    if len(gsm_samples) != n_gsm:
        raise RuntimeError(f"Collected {len(gsm_samples)} GSM8K samples, expected {n_gsm}.")

    # Combine and shuffle
    all_samples = c4_samples + gsm_samples
    indices = torch.randperm(len(all_samples), generator=generator).tolist()
    shuffled_samples = [all_samples[i] for i in indices]

    print(f"Collected {len(shuffled_samples)} hybrid calibration sequences of length {sequence_length}.")
    return shuffled_samples


def iter_batches(samples, batch_size):
    """Join token sequences into small batches."""
    for start in range(0, len(samples), batch_size):
        yield torch.cat(samples[start : start + batch_size], dim=0)


def save_model_and_tokenizer(model, tokenizer, output_dir):
    """Save model and tokenizer in normal Hugging Face format."""
    path = Path(output_dir)
    path.mkdir(parents=True, exist_ok=True)
    print(f"\nSaving model to {path}")
    model.save_pretrained(path, safe_serialization=True)
    tokenizer.save_pretrained(path)


In [22]:

# Configurable Pruning Ratios
MODEL_ID = "Qwen/Qwen3.5-0.8B"
OUTPUT_DIR = "models/qwen-width-pruned"
ATTN_HEAD_PRUNE_RATIO = 0.25   # Prune 25% of attention heads (leaves 6 out of 8 heads)
MLP_PRUNE_RATIO = 0.20         # Prune 20% of MLP intermediate dimension (leaves 2864 out of 3584)

N_CALIBRATION_SAMPLES = 32      # Set to 128 for reliable pruning, or 4 for a quick smoke test
SEQUENCE_LENGTH = 2048          # Set to 2048 for full context, or 64 for a quick smoke test
RANDOM_SEED = 0
BATCH_SIZE = 1
TORCH_DTYPE = "auto"
DEVICE_MAP = "auto"
TRUST_REMOTE_CODE = True       # Qwen3.5 text model needs trust_remote_code=True for hybrid layers

# --- Activation Stats Collector ---

class ActivationStatsCollector:
    """Accumulates activation norms for attention heads and MLP intermediate dimensions."""
    def __init__(self, model):
        self.model = model
        # layer_idx -> sum of squared activations for query heads [num_attention_heads]
        self.attn_stats = defaultdict(lambda: None)
        # layer_idx -> sum of squared activations for MLP intermediate dimensions [intermediate_size]
        self.mlp_stats = defaultdict(lambda: None)
        self.handles = []

    def register_hooks(self):
        layers = get_decoder_layers(self.model)
        for layer_idx, layer in enumerate(layers):
            # 1. Standard self_attn layer (Qwen3_5Attention)
            if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "o_proj"):
                o_proj = layer.self_attn.o_proj
                handle = o_proj.register_forward_hook(
                    self.make_attn_hook(layer_idx, layer.self_attn)
                )
                self.handles.append(handle)
            
            # 2. MLP layer (Qwen3_5MLP)
            if hasattr(layer, "mlp") and hasattr(layer.mlp, "down_proj"):
                down_proj = layer.mlp.down_proj
                handle = down_proj.register_forward_hook(
                    self.make_mlp_hook(layer_idx)
                )
                self.handles.append(handle)

    def make_attn_hook(self, layer_idx, self_attn):
        num_heads = self_attn.config.num_attention_heads
        head_dim = self_attn.head_dim
        def hook(module, inputs, output):
            # Input to o_proj is the concatenated outputs of the heads.
            # Shape: [batch, seq, num_attention_heads * head_dim]
            x = inputs[0].detach().float()
            x = x.reshape(-1, num_heads, head_dim)
            # Sum squares over the token (batch * seq) and head_dim dimensions
            scores = torch.sum(x.pow(2), dim=(0, 2)).cpu()
            if self.attn_stats[layer_idx] is None:
                self.attn_stats[layer_idx] = scores
            else:
                self.attn_stats[layer_idx] += scores
        return hook

    def make_mlp_hook(self, layer_idx):
        def hook(module, inputs, output):
            # Input to down_proj is the intermediate activations.
            # Shape: [batch, seq, intermediate_size]
            x = inputs[0].detach().float()
            x = x.reshape(-1, x.shape[-1])
            # Sum squares over the token dimension
            scores = torch.sum(x.pow(2), dim=0).cpu()
            if self.mlp_stats[layer_idx] is None:
                self.mlp_stats[layer_idx] = scores
            else:
                self.mlp_stats[layer_idx] += scores
        return hook

    def remove_hooks(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

# --- GQA Compatibility Resolvers ---

def get_valid_gqa_targets(num_heads, num_kv_heads, prune_ratio):
    """Calculate valid GQA target head numbers that are divisible and <= original."""
    target_heads = max(1, int(round(num_heads * (1.0 - prune_ratio))))
    target_kv_heads = max(1, int(round(num_kv_heads * (1.0 - prune_ratio))))
    
    if target_heads % target_kv_heads == 0:
        return target_heads, target_kv_heads
        
    best_h, best_kv = None, None
    min_dist = float('inf')
    for h in range(1, num_heads + 1):
        for kv in range(1, num_kv_heads + 1):
            if h % kv == 0:
                dist = abs(h - target_heads) + abs(kv - target_kv_heads)
                if dist < min_dist:
                    min_dist = dist
                    best_h = h
                    best_kv = kv
    return best_h, best_kv

def select_heads_to_keep(scores, num_heads, num_kv_heads, target_heads, target_kv_heads):
    """Select the most important query heads and key-value heads while preserving symmetry."""
    orig_group_size = num_heads // num_kv_heads
    new_group_size = target_heads // target_kv_heads
    
    # Calculate group importance scores
    kv_scores = []
    for k in range(num_kv_heads):
        group_query_heads = range(k * orig_group_size, (k + 1) * orig_group_size)
        group_score = sum(scores[h].item() for h in group_query_heads)
        kv_scores.append((k, group_score))
        
    # Keep target_kv_heads KV groups with the highest aggregated scores
    kv_scores.sort(key=lambda x: x[1], reverse=True)
    kept_kv_indices = [kv_scores[i][0] for i in range(target_kv_heads)]
    kept_kv_indices.sort()
    
    # Within each kept KV group, keep the top new_group_size query heads
    kept_query_indices = []
    for old_k_idx in kept_kv_indices:
        group_query_heads = list(range(old_k_idx * orig_group_size, (old_k_idx + 1) * orig_group_size))
        group_query_heads.sort(key=lambda h: scores[h].item(), reverse=True)
        selected_heads = group_query_heads[:new_group_size]
        selected_heads.sort()
        kept_query_indices.extend(selected_heads)
        
    return kept_query_indices, kept_kv_indices

# --- Layer Weight Slicing ---

def prune_attn_layer(self_attn, kept_q_indices, kept_kv_indices):
    """Replace linear modules in self_attn with sliced ones."""
    head_dim = self_attn.head_dim
    device = self_attn.q_proj.weight.device
    dtype = self_attn.q_proj.weight.dtype

    # 1. q_proj (output features: num_attention_heads * head_dim * 2)
    # The output contains both standard queries and gates concatenated.
    q_indices = []
    for h in kept_q_indices:
        q_indices.extend(list(range(h * head_dim * 2, (h + 1) * head_dim * 2)))
    q_proj = self_attn.q_proj
    new_q_weight = q_proj.weight.data[q_indices, :]
    new_q_proj = nn.Linear(q_proj.in_features, len(q_indices), bias=(q_proj.bias is not None)).to(device=device, dtype=dtype)
    new_q_proj.weight.data.copy_(new_q_weight)
    if q_proj.bias is not None:
        new_q_proj.bias.data.copy_(q_proj.bias.data[q_indices])
    self_attn.q_proj = new_q_proj

    # 2. k_proj (output features: num_key_value_heads * head_dim)
    k_indices = []
    for k in kept_kv_indices:
        k_indices.extend(list(range(k * head_dim, (k + 1) * head_dim)))
    k_proj = self_attn.k_proj
    new_k_weight = k_proj.weight.data[k_indices, :]
    new_k_proj = nn.Linear(k_proj.in_features, len(k_indices), bias=(k_proj.bias is not None)).to(device=device, dtype=dtype)
    new_k_proj.weight.data.copy_(new_k_weight)
    if k_proj.bias is not None:
        new_k_proj.bias.data.copy_(k_proj.bias.data[k_indices])
    self_attn.k_proj = new_k_proj

    # 3. v_proj (output features: num_key_value_heads * head_dim)
    v_proj = self_attn.v_proj
    new_v_weight = v_proj.weight.data[k_indices, :]
    new_v_proj = nn.Linear(v_proj.in_features, len(k_indices), bias=(v_proj.bias is not None)).to(device=device, dtype=dtype)
    new_v_proj.weight.data.copy_(new_v_weight)
    if v_proj.bias is not None:
        new_v_proj.bias.data.copy_(v_proj.bias.data[k_indices])
    self_attn.v_proj = new_v_proj

    # 4. o_proj (input features: num_attention_heads * head_dim)
    o_indices = []
    for h in kept_q_indices:
        o_indices.extend(list(range(h * head_dim, (h + 1) * head_dim)))
    o_proj = self_attn.o_proj
    new_o_weight = o_proj.weight.data[:, o_indices]
    new_o_proj = nn.Linear(len(o_indices), o_proj.out_features, bias=(o_proj.bias is not None)).to(device=device, dtype=dtype)
    new_o_proj.weight.data.copy_(new_o_weight)
    if o_proj.bias is not None:
        new_o_proj.bias.data.copy_(o_proj.bias.data)
    self_attn.o_proj = new_o_proj

    # Update groups count
    self_attn.num_key_value_groups = len(kept_q_indices) // len(kept_kv_indices)

def prune_mlp_layer(mlp, kept_mlp_indices):
    """Replace linear modules in mlp with sliced ones."""
    device = mlp.gate_proj.weight.device
    dtype = mlp.gate_proj.weight.dtype
    
    # 1. gate_proj (output features: intermediate_size)
    gate_proj = mlp.gate_proj
    new_gate_weight = gate_proj.weight.data[kept_mlp_indices, :]
    new_gate_proj = nn.Linear(gate_proj.in_features, len(kept_mlp_indices), bias=(gate_proj.bias is not None)).to(device=device, dtype=dtype)
    new_gate_proj.weight.data.copy_(new_gate_weight)
    if gate_proj.bias is not None:
        new_gate_proj.bias.data.copy_(gate_proj.bias.data[kept_mlp_indices])
    mlp.gate_proj = new_gate_proj

    # 2. up_proj (output features: intermediate_size)
    up_proj = mlp.up_proj
    new_up_weight = up_proj.weight.data[kept_mlp_indices, :]
    new_up_proj = nn.Linear(up_proj.in_features, len(kept_mlp_indices), bias=(up_proj.bias is not None)).to(device=device, dtype=dtype)
    new_up_proj.weight.data.copy_(new_up_weight)
    if up_proj.bias is not None:
        new_up_proj.bias.data.copy_(up_proj.bias.data[kept_mlp_indices])
    mlp.up_proj = new_up_proj

    # 3. down_proj (input features: intermediate_size)
    down_proj = mlp.down_proj
    new_down_weight = down_proj.weight.data[:, kept_mlp_indices]
    new_down_proj = nn.Linear(len(kept_mlp_indices), down_proj.out_features, bias=(down_proj.bias is not None)).to(device=device, dtype=dtype)
    new_down_proj.weight.data.copy_(new_down_weight)
    if down_proj.bias is not None:
        new_down_proj.bias.data.copy_(down_proj.bias.data)
    mlp.down_proj = new_down_proj

# --- Main Pruning Driver ---

def apply_width_pruning(model, samples):
    """Executes the entire activation collection and width pruning pipeline."""
    # Step 1: Collect activation statistics
    collector = ActivationStatsCollector(model)
    collector.register_hooks()
    
    device = get_input_device(model)
    print("Collecting activation statistics on calibration dataset...")
    with torch.no_grad():
        for i, batch in enumerate(iter_batches(samples, BATCH_SIZE)):
            batch = batch.to(device)
            # Run forward pass of the model
            model(batch)
            if i % 10 == 0:
                print(f"  Processed batch {i+1}")
            
    collector.remove_hooks()
    
    # Step 2: Determine target shapes
    config = model.config.text_config if hasattr(model.config, "text_config") else model.config
    target_heads, target_kv_heads = get_valid_gqa_targets(
        config.num_attention_heads,
        config.num_key_value_heads,
        ATTN_HEAD_PRUNE_RATIO
    )
    
    target_intermediate_size = int(round(config.intermediate_size * (1.0 - MLP_PRUNE_RATIO) / 8) * 8)
    target_intermediate_size = max(8, target_intermediate_size)
    
    print(f"\nTarget Attention Query Heads: {target_heads} (original {config.num_attention_heads})")
    print(f"Target Attention KV Heads: {target_kv_heads} (original {config.num_key_value_heads})")
    print(f"Target MLP Intermediate Size: {target_intermediate_size} (original {config.intermediate_size})")
    
    # Step 3: Physically prune layers
    layers = get_decoder_layers(model)
    for layer_idx, layer in enumerate(layers):
        # 1. Prune standard attention heads
        if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "o_proj"):
            scores = collector.attn_stats[layer_idx]
            if scores is not None:
                kept_q, kept_kv = select_heads_to_keep(
                    scores,
                    config.num_attention_heads,
                    config.num_key_value_heads,
                    target_heads,
                    target_kv_heads
                )
                print(f"Layer {layer_idx:02d} standard attention: keeping Q-heads {kept_q}, KV-heads {kept_kv}")
                prune_attn_layer(layer.self_attn, kept_q, kept_kv)
                
        # 2. Prune MLP intermediate dimension
        if hasattr(layer, "mlp") and hasattr(layer.mlp, "down_proj"):
            scores = collector.mlp_stats[layer_idx]
            if scores is not None:
                sorted_indices = torch.argsort(scores, descending=True)
                kept_mlp = sorted_indices[:target_intermediate_size].tolist()
                kept_mlp.sort() # keep original relative order
                prune_mlp_layer(layer.mlp, kept_mlp)
                
    # Step 4: Update global configuration
    if hasattr(model.config, "text_config") and model.config.text_config is not None:
        model.config.text_config.num_attention_heads = target_heads
        model.config.text_config.num_key_value_heads = target_kv_heads
        model.config.text_config.intermediate_size = target_intermediate_size
        
    model.config.num_attention_heads = target_heads
    model.config.num_key_value_heads = target_kv_heads
    model.config.intermediate_size = target_intermediate_size
    
    print("\nModel successfully pruned!")

# --- Main Entry Point ---

def main():
    model, tokenizer = load_model_and_tokenizer(
        MODEL_ID,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=TRUST_REMOTE_CODE,
    )
    
    # Run a pre-pruning test generation
    print("\nPre-pruning test generation:")
    device = get_input_device(model)
    inputs = tokenizer("Pruning language models is", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=30)
    print(f"Generated: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

    # Load calibration data
    samples = load_c4_calibration(
        tokenizer,
        n_samples=N_CALIBRATION_SAMPLES,
        sequence_length=SEQUENCE_LENGTH,
        seed=RANDOM_SEED,
    )
    
    # Prune
    apply_width_pruning(model, samples)
    
    # Post-pruning test generation
    print("\nPost-pruning test generation:")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=30)
    print(f"Generated: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
    
    # Save the pruned model and tokenizer
    save_model_and_tokenizer(model, tokenizer, OUTPUT_DIR)

if __name__ == "__main__":
    main()


Loading tokenizer: Qwen/Qwen3.5-0.8B
Loading model: Qwen/Qwen3.5-0.8B


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



Pre-pruning test generation:
Generated: Pruning language models is a technique that involves removing a portion of the model's parameters to reduce its size and improve performance. This process is often used to address the limitations of
Loading C4 calibration data from allenai/c4 first train shard (samples=32, sequence_length=2048)...
  collected calibration sample 10/32
  collected calibration sample 20/32
  collected calibration sample 30/32
Collected 32 calibration sequences of length 2048.
  Processed batch 1
  Processed batch 11
  Processed batch 21
  Processed batch 31

Target Attention Query Heads: 6 (original 8)
Target Attention KV Heads: 2 (original 2)
Target MLP Intermediate Size: 2864 (original 3584)
Layer 03 standard attention: keeping Q-heads [0, 1, 3, 4, 5, 7], KV-heads [0, 1]
Layer 07 standard attention: keeping Q-heads [0, 2, 3, 4, 6, 7], KV-heads [0, 1]
Layer 11 standard attention: keeping Q-heads [0, 1, 3, 4, 6, 7], KV-heads [0, 1]
Layer 15 standard attention: keep

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Layer 23 standard attention: keeping Q-heads [0, 1, 2, 5, 6, 7], KV-heads [0, 1]

Model successfully pruned!

Post-pruning test generation:
Generated: Pruning language models is a common practice in the industry.
The following are the most common practices:
1. 1.
2. 2.
3

Saving model to models/qwen-width-pruned


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
def count_parameters(model):
    # Sums up the number of elements in all parameters of the model
    return sum(p.numel() for p in model.parameters())
# 1. Load the original model on meta device to check its size without consuming memory
print("Loading original model...")
orig_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    trust_remote_code=True,
    device_map="meta"
)
orig_params = count_parameters(orig_model)
# 2. Load the pruned model on meta device
print("Loading pruned model...")
pruned_model = AutoModelForCausalLM.from_pretrained(
    "models/qwen-width-pruned",
    trust_remote_code=True,
    device_map="meta"
)
pruned_params = count_parameters(pruned_model)
# 3. Print the comparison
print(f"\nParameter Count Comparison:")
print(f"  Original model: {orig_params:,} parameters")
print(f"  Pruned model:   {pruned_params:,} parameters")
print(f"  Difference:     {orig_params - pruned_params:,} parameters pruned")
print(f"  Reduction:      {(1.0 - pruned_params / orig_params) * 100:.2f}% of total weights removed")

Loading original model...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading pruned model...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


Parameter Count Comparison:
  Original model: 752,393,024 parameters
  Pruned model:   689,871,680 parameters
  Difference:     62,521,344 parameters pruned
  Reduction:      8.31% of total weights removed


In [ ]:
!lm_eval --model hf \
    --model_args pretrained=Qwen/Qwen3.5-0.8B \
    --tasks mmlu,gsm8k \
    --device cuda:0 \
    --batch_size 1 \
    --num_fewshot 5 \
    --limit 1 \
    --verbosity WARNING \
    --output_path baseline_results.json

!lm_eval --model hf \
    --model_args pretrained=models/qwen-width-pruned \
    --tasks mmlu,gsm8k \
    --device cuda:0 \
    --batch_size 1 \
    --num_fewshot 5 \
    --limit 1 \
    --verbosity WARNING \
    --output_path pruned_results.json

2026-06-12:14:57:14 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-12:14:57:25 INFO     [_cli.run:388] Selected Tasks: ['mmlu', 'gsm8k']
2026-06-12:14:57:28 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-06-12:14:57:28 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen3.5-0.8B'}
2026-06-12:14:57:33 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-06-12:14:57:35 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/c